In [1]:
import torch
from torch.utils import data
import re
from torch import nn
from torch.nn import functional as F
import math
import random
batch_size=32
num_step=35
lr=1
epoch=500

In [2]:
def corpus_vocab_maker():
    with open('../data/timemachine.txt','r') as file:
        text=file.read().lower()
    text=re.sub('[^A-Za-z]+',' ',text).strip()
    tokens=list(text)
    idx_to_token=[*(['<unk>']+list(set(tokens)))]
    token_to_idx={
        token:idx for idx,token in enumerate(idx_to_token)
    }
    vocab={
        "idx_to_token":idx_to_token,
        "token_to_idx":token_to_idx
    }
    corpus=torch.tensor([token_to_idx[token] for token in tokens])
    return corpus,vocab

将文件读进去，转成小写，将连续的非英文字母替换成一个空格。接着建立字典，将文本转为数字

In [3]:
def train_iter_maker(corpus,batch_size,num_step):
    t = random.randint(
        0,
        num_step - 1
    )
    corpus = corpus[t:]
    sequence=corpus.unfold(dimension=0,size=num_step+1,step=num_step)
    x = sequence[:, :-1]
    y = sequence[:, 1:]
    dataset=data.TensorDataset(x,y)
    train_iter=data.DataLoader(dataset,batch_size=batch_size,shuffle=True)
    return train_iter

# class SequentialTextLoader:
#     def __init__(
#         self,
#         corpus,
#         batch_size,
#         num_step
#     ):
#         self.corpus = (
#             corpus
#             .flatten()
#             .long()
#         )

#         self.batch_size = batch_size
#         self.num_step = num_step

#     def __iter__(self):
#         # 每次进入一个新epoch时，
#         # 都重新产生随机切片偏移
#         t = random.randint(
#             0,
#             self.num_step - 1
#         )

#         # 减1是因为还需要给X制作错开一位的Y
#         num_tokens = (
#             (
#                 len(self.corpus)
#                 - t
#                 - 1
#             )
#             // self.batch_size
#         ) * self.batch_size

#         if num_tokens <= 0:
#             raise ValueError(
#                 "corpus长度不足以构造一个batch"
#             )

#         x_all = self.corpus[
#             t:
#             t + num_tokens
#         ]

#         y_all = self.corpus[
#             t + 1:
#             t + 1 + num_tokens
#         ]

#         # 每一行都是一条连续的文本序列
#         x_all = x_all.reshape(
#             self.batch_size,
#             -1
#         )

#         y_all = y_all.reshape(
#             self.batch_size,
#             -1
#         )

#         num_batches = (
#             x_all.shape[1]
#             // self.num_step
#         )

#         for start in range(
#             0,
#             num_batches * self.num_step,
#             self.num_step
#         ):
#             x = x_all[
#                 :,
#                 start:start + self.num_step
#             ]

#             y = y_all[
#                 :,
#                 start:start + self.num_step
#             ]

#             yield x, y


# def train_iter_maker(
#     corpus,
#     batch_size,
#     num_step
# ):
#     train_iter = SequentialTextLoader(
#         corpus,
#         batch_size,
#         num_step
#     )

#     return train_iter

In [4]:
corpus,vocab=corpus_vocab_maker()
# corpus=corpus[:10000]
device=torch.device('cuda:0')
vocab_size=len(vocab['idx_to_token'])



In [5]:
class rnnnet(nn.Module):
    def __init__(self,input_num,hidden_num,layer_num):
        super().__init__()
        self.input_num=input_num
        self.hidden_num=hidden_num
        self.layer_num=layer_num
        self.rnn=nn.RNN(input_size=input_num,hidden_size=hidden_num,num_layers=layer_num)
        self.output=nn.Linear(hidden_num,input_num)
    def forward(self,x,state):
        x=F.one_hot(x.T.long(),num_classes=self.input_num).float() #转成独热编码，相当于做分类预测 原本x=[batch,time_step长度的corpus]  
        rnn_output,now_state=self.rnn(x,state) #rnn_out=[time_step,batch,hidden_num]
        rnn_output=rnn_output.reshape(-1,self.hidden_num)#展成[time_step*batch,hidden_num]
        result=self.output(rnn_output)#result=[time_step*batch,output_num]
        return result,now_state
    def begin_state(self,batch_size,device): #多一个layer_num主要是为了多层rnn准备的
        return torch.zeros(self.layer_num,batch_size,self.hidden_num,device=device)
        



nn.RNN返回的结果是映射到隐藏层的输出，以及根据历史信息更新的state，也就是课程里的h，最后需要添加一个线性层，将隐藏层的输出映射到线性层上

定义这个rnnnet需要的参数有：输入维度(输出维度)，隐藏层个数，rnn层数

对于每个时间步 \(t\)，RNN根据当前输入 \(x_t\) 和上一个隐藏状态 \(h_{t-1}\)，计算新的隐藏状态：

$$
h_t = \tanh
\left(
W_{xh}x_t + W_{hh}h_{t-1} + b_h
\right)
$$

其中：

- \(x_t\)：当前时间步输入的字符
- \(h_{t-1}\)：此前历史信息的总结
- \(h_t\)：更新后的隐藏状态

`nn.RNN` 返回：

$$
H=(h_1,h_2,\ldots,h_T)
$$

以及序列结束时的最终隐藏状态：

$$
h_T
$$


In [6]:
net=rnnnet(vocab_size,256,1).to(device)
trainer=torch.optim.SGD(net.parameters(),lr=lr)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(trainer,T_max=epoch,eta_min=1e-4)
loss = nn.CrossEntropyLoss()
for i in range(epoch):
    train_iter=train_iter_maker(corpus,batch_size,num_step)
    net.train()
    train_loss=0.0
    train_count=0.0
    state=None
    for x,y in train_iter:
        x=x.to(device)
        y=(y.T.reshape(-1).long().to(device))  ## rnn_output按照“时间优先”展开， 所以标签也要先转置再展开，保证预测和标签顺序一致
        # if state is None:
        #     state = net.begin_state(
        #         x.shape[0],
        #         device
        #     )
        # else:
        #     # 保留隐藏状态数值，但断开之前的计算图
        #     state = state.detach()
        state=net.begin_state(x.shape[0],device) #需要初始化隐藏状态，但是由于每个batch的信息不一定连续，所以每个batch都要初始化
        trainer.zero_grad()
        logit,state=net(x,state)
        l=loss(logit,y)
        l.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(),max_norm=1.0) #梯度裁剪 防止梯度过大
        trainer.step()
        train_loss+=l.item()*len(y)
        train_count+=len(y)
    scheduler.step()
    print(f"第{i+1}epoch的loss:{train_loss/train_count},困惑度：{math.exp(train_loss/train_count)}")


第1epoch的loss:2.476420870893753,困惑度：11.898601484422441
第2epoch的loss:2.2076229456215075,困惑度：9.094073572746364
第3epoch的loss:2.1440086636179436,困惑度：8.533577396988102
第4epoch的loss:2.0948550270890633,困惑度：8.124263086765191
第5epoch的loss:2.0449737067370823,困惑度：7.72895531578401
第6epoch的loss:1.9985397912631075,困惑度：7.378274408332743
第7epoch的loss:1.956260059310103,困惑度：7.072825590855821
第8epoch的loss:1.9198506839053016,困惑度：6.819940066443875
第9epoch的loss:1.8837258701705624,困惑度：6.577967915249768
第10epoch的loss:1.8531352765241553,困惑度：6.379790606524628
第11epoch的loss:1.8277656147929513,困惑度：6.219973304077678
第12epoch的loss:1.800447550039403,困惑度：6.052355590339748
第13epoch的loss:1.7762622071891048,困惑度：5.9077332156316915
第14epoch的loss:1.7526923996044061,困惑度：5.770117242308101
第15epoch的loss:1.7327612905575471,困惑度：5.656250916829826
第16epoch的loss:1.7149828987637499,困惑度：5.556580487153243
第17epoch的loss:1.6994004584051785,困惑度：5.470666516186948
第18epoch的loss:1.6826159621035808,困惑度：5.379610437167492
第19epoch的loss:1.67073

In [7]:
def predict_text(text,num_letter,net,vocab,device):
    net.eval()
    state=net.begin_state(1,device) #预测的时候每次传进去1个字符，吐出一个字符
    output_idx=[vocab['token_to_idx'][text[0]]] #将第一个字符先扔进去，用于传入net
    def get_input():
        return torch.tensor(output_idx[-1],dtype=torch.long,device=device).reshape(1,-1) #将当前已读的字符最后一个弄成tensor传入net
    with torch.no_grad():
        for token in text[1:]:
            _,state=net(get_input(),state) #这个过程预热，主要用于更新state
            output_idx.append(vocab['token_to_idx'][token])
        for i in range(num_letter):
            y,state=net(get_input(),state)
            y=y[-1].argmax(dim=-1).item()
            output_idx.append(y)

        return "".join([
        vocab['idx_to_token'][index]
        for index in output_idx])
        

In [8]:
predict_text("world",100,net,vocab,device)

'world there was a little people and the same to the same seat of my confusion the same seat of my confusi'